# Add ImpactMe outcome probs via saved LR models

In [1]:
import os
import sys

sys.path.append(os.path.abspath("../"))
from src.config import BASE_PATH, SEED
from src.utils import col_config_sensitivity
from src.utils import ALPHA_LIST
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from shutil import rmtree
import joblib
import torch

In [ ]:
SEG_TYPES = [
    "turns",
    "pt_noshort",
]
MODEL_NAMES = [
    "Llama-3.3-8B-Instruct",
    "Llama-3.3-70B-Instruct",
]
IMPACT_TYPES = ["self", "parent"]

Get Probs

In [ ]:
def add_prob_vector(embed_dir, impact_type, model_name, impactme_seg_type):
    ## ========> Validate input
    if impact_type not in ["self", "parent"]:
        raise ValueError(
            f"Impact type must be one of ['self', 'parent'], got {impact_type}"
        )
    if impactme_seg_type not in ["turns", "pt_noshort"]:
        raise ValueError(
            f"Seg type must be one of ['turns', 'pt_noshort'], got {impactme_seg_type}"
        )
    if model_name not in ["Llama-3.3-8B-Instruct", "Llama-3.3-70B-Instruct"]:
        raise ValueError(
            f"model_name must be one of ['Llama-3.3-8B-Instruct', 'Llama-3.3-70B-Instruct'], got {model_name}"
        )
    ## ==========? Load df w/ text embeddings
    embed_df = pd.read_parquet(embed_dir / model_name / f"{impact_type}.parquet")
    ## ============> Load ImpactMe LR models
    lr_model_dir = Path(
        f"/lscratch/mataray/trained_lr_models/{impactme_seg_type}/{model_name}"
    )
    lr_model_dict = {}
    for outcome_file in lr_model_dir.iterdir():
        outcome_name = outcome_file.stem
        # Skips `any` and any comprehensive `a` or `c` outcomes
        if len(outcome_name) != 2:
            continue
        lr_model_dict[outcome_name] = joblib.load(outcome_file)
    outcome_order = sorted(lr_model_dict.keys())
    if len(outcome_order) != 28:
        raise ValueError(
            f"outcome_order should be length 28, got {len(outcome_order)}. \nOutcomes: {outcome_order}"
        )
    proba_dict = {}
    ## ==========> Get probs
    X = np.stack(embed_df["embedding"].values).astype(np.float16)
    if not torch.is_tensor(X):
        X = torch.from_numpy(X)
    for outcome_name in outcome_order:
        lr_model = lr_model_dict[outcome_name]
        y_proba = lr_model.predict_proba(X)[:, 1]
        proba_dict[outcome_name] = y_proba
    ### Ensure right number of probs
    probs_df = pd.DataFrame(proba_dict)
    # Ensure correctly ordered
    probs_df = probs_df[outcome_order]
    # Validate number of rows matches og df
    if len(probs_df) != len(embed_df):
        raise ValueError("Number of probability rows doesn't match embed_df length")
    ### Format probs + add to df
    embed_df["prob_vector"] = [row.values for _, row in probs_df.iterrows()]
    ## ========> EXPORT
    proba_export_path = (
        BASE_PATH
        / "data"
        / "cleaned"
        / "block"
        / "w_probs"
        / impactme_seg_type
        / model_name
        / f"{impact_type}.parquet"
    )
    if proba_export_path.exists():
        proba_export_path.unlink()
    proba_export_path.parent.mkdir(exist_ok=True, parents=True)
    embed_df.to_parquet(proba_export_path)

In [ ]:
embed_dir = BASE_PATH / "data" / "cleaned" / "block" / "w_embeddings"
for impact_type in IMPACT_TYPES:
    for seg_type in SEG_TYPES:
        for model_name in MODEL_NAMES:
            add_prob_vector(
                embed_dir=embed_dir,
                impact_type="self",
                model_name=model_name,
                impactme_seg_type=seg_type,
            )
            add_prob_vector(
                embed_dir=embed_dir,
                impact_type="parent",
                model_name=model_name,
                impactme_seg_type=seg_type,
            )